In [0]:
CATALOG = "databricks_rr"     
SCHEMA = "default"        
VOLUME = "capstone"        

DATA_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/project1"     
OUTPUT_PATH = f"{DATA_PATH}/output"                              

In [0]:
# create the volume once (it is safe to run again)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print("Volume ready. Upload attendance.csv and tasks.csv into the folder project1 of this volume.")

Volume ready. Upload attendance.csv and tasks.csv into the folder project1 of this volume.


In [0]:
# all tables below will be created in this catalog and schema
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

from pyspark.sql import functions as F
import os

## Step 1: Extract - load the csv files

In [0]:
# read everything as text first, then change the data types while cleaning
attendance_raw = spark.read.csv(f"{DATA_PATH}/attendance.csv", header=True)
tasks_raw = spark.read.csv(f"{DATA_PATH}/tasks.csv", header=True)

print("Attendance rows:", attendance_raw.count())
print("Task rows:", tasks_raw.count())
display(attendance_raw.limit(5))
display(tasks_raw.limit(5))

Attendance rows: 402
Task rows: 400


employee_id,employee_name,department,work_date,clock_in,clock_out,break_minutes,status
1,Employee 1,Engineering,2025-01-06,2025-01-06 09:23:46,2025-01-06 18:19:25,34.0,LATE
1,Employee 1,Engineering,2025-01-07,null,null,0.0,ABSENT
1,Employee 1,Engineering,2025-01-08,2025-01-08 09:07:18,2025-01-08 17:35:32,33.0,PRESENT
1,Employee 1,Engineering,2025-01-09,2025-01-09 09:02:00,2025-01-09 17:23:12,52.0,PRESENT
1,Employee 1,Engineering,2025-01-10,2025-01-10 09:14:01,2025-01-10 19:19:58,48.0,PRESENT


employee_id,work_date,tasks_assigned,tasks_completed
1,2025-01-06,5,4.0
1,2025-01-07,0,0.0
1,2025-01-08,6,5.0
1,2025-01-09,8,8.0
1,2025-01-10,4,3.0


## Step 2: Transform - clean the attendance data

In [0]:
# remove duplicate rows
attendance = attendance_raw.dropDuplicates()

# change data types (try_cast and try_to_timestamp give null for wrong values instead of error)
attendance = attendance.withColumn("break_minutes", F.expr("try_cast(break_minutes as double)"))
attendance = attendance.withColumn("clock_in", F.expr("try_to_timestamp(clock_in, 'yyyy-MM-dd HH:mm:ss')"))
attendance = attendance.withColumn("clock_out", F.expr("try_to_timestamp(clock_out, 'yyyy-MM-dd HH:mm:ss')"))

# negative break time is wrong, so make it null
attendance = attendance.withColumn(
    "break_minutes",
    F.when(F.col("break_minutes") >= 0, F.col("break_minutes")))

# fill missing break time with the median break
median_break = attendance.agg(F.percentile_approx("break_minutes", 0.5)).collect()[0][0]
attendance = attendance.fillna({"break_minutes": median_break})
print("Median break used:", median_break)
print("Rows after removing duplicates:", attendance.count())

Median break used: 48.0
Rows after removing duplicates: 400


In [0]:
# clean the tasks data
tasks = tasks_raw.dropDuplicates(["employee_id", "work_date"])
tasks = tasks.withColumn("tasks_assigned", F.expr("try_cast(tasks_assigned as int)"))
tasks = tasks.withColumn("tasks_completed", F.expr("try_cast(tasks_completed as double)"))

## Step 3: Transform - combine attendance and tasks, calculate work hours and productivity

In [0]:
# separate absent rows and working rows (PRESENT or LATE)
absent = attendance.filter(F.col("status") == "ABSENT")
worked = attendance.filter(F.col("status").isin("PRESENT", "LATE"))

# remove working rows with missing time or clock out before clock in
worked = worked.dropna(subset=["clock_in", "clock_out"])
worked = worked.filter(F.col("clock_out") > F.col("clock_in"))

# work hours = (clock out - clock in) in hours - break time
worked = worked.withColumn(
    "work_hours",
    F.round((F.unix_timestamp("clock_out") - F.unix_timestamp("clock_in")) / 3600
            - F.col("break_minutes") / 60, 2))
worked = worked.filter(F.col("work_hours") > 0)

# combine with tasks data (join on employee and date)
worked = worked.join(tasks, on=["employee_id", "work_date"], how="left")

# productivity = tasks completed per work hour
worked = worked.withColumn(
    "productivity_score",
    F.round(F.col("tasks_completed") / F.col("work_hours"), 2))

# completion rate = tasks completed out of tasks assigned
worked = worked.withColumn(
    "completion_pct",
    F.when(F.col("tasks_assigned") > 0,
           F.round(F.col("tasks_completed") / F.col("tasks_assigned") * 100, 2)))

# put absent rows back so the clean table has every valid record
# (absent rows have no work hours or tasks, so those columns stay empty)
attendance_clean = worked.unionByName(absent, allowMissingColumns=True)
attendance_clean = attendance_clean.select(
    "employee_id", "employee_name", "department", "work_date", "status",
    "clock_in", "clock_out", "break_minutes", "work_hours",
    "tasks_assigned", "tasks_completed", "productivity_score", "completion_pct")

print("Working rows:", worked.count())
print("Absent rows:", absent.count())
display(attendance_clean.orderBy("employee_id", "work_date").limit(10))

Working rows: 357
Absent rows: 40


employee_id,employee_name,department,work_date,status,clock_in,clock_out,break_minutes,work_hours,tasks_assigned,tasks_completed,productivity_score,completion_pct
1,Employee 1,Engineering,2025-01-06,LATE,2025-01-06T09:23:46.000Z,2025-01-06T18:19:25.000Z,34.0,8.36,5,4.0,0.48,80.0
1,Employee 1,Engineering,2025-01-07,ABSENT,null,null,0.0,null,null,null,null,null
1,Employee 1,Engineering,2025-01-08,PRESENT,2025-01-08T09:07:18.000Z,2025-01-08T17:35:32.000Z,33.0,7.92,6,5.0,0.63,83.33
1,Employee 1,Engineering,2025-01-09,PRESENT,2025-01-09T09:02:00.000Z,2025-01-09T17:23:12.000Z,52.0,7.49,8,8.0,1.07,100.0
1,Employee 1,Engineering,2025-01-10,PRESENT,2025-01-10T09:14:01.000Z,2025-01-10T19:19:58.000Z,48.0,9.3,4,3.0,0.32,75.0
1,Employee 1,Engineering,2025-01-14,LATE,2025-01-14T09:17:06.000Z,2025-01-14T18:24:19.000Z,64.0,8.05,6,6.0,0.75,100.0
1,Employee 1,Engineering,2025-01-15,PRESENT,2025-01-15T09:09:38.000Z,2025-01-15T17:54:49.000Z,58.0,7.79,8,8.0,1.03,100.0
1,Employee 1,Engineering,2025-01-16,PRESENT,2025-01-16T09:08:56.000Z,2025-01-16T17:57:05.000Z,51.0,7.95,4,4.0,0.5,100.0
1,Employee 1,Engineering,2025-01-17,PRESENT,2025-01-17T09:01:24.000Z,2025-01-17T17:10:34.000Z,41.0,7.47,7,6.0,0.8,85.71
1,Employee 1,Engineering,2025-01-20,LATE,2025-01-20T09:21:33.000Z,2025-01-20T16:46:52.000Z,55.0,6.51,4,4.0,0.61,100.0


## Step 4: Transform - department level KPIs

In [0]:
# total records per department (all rows after removing duplicates)
dept_total = attendance.groupBy("department").agg(
    F.count("*").alias("total_records"))

# absent and late counts (from the clean table)
dept_issues = attendance_clean.groupBy("department").agg(
    F.sum(F.when(F.col("status") == "ABSENT", 1).otherwise(0)).alias("absent_count"),
    F.sum(F.when(F.col("status") == "LATE", 1).otherwise(0)).alias("late_login_count"))

# work and task KPIs (only from working rows)
dept_work = worked.groupBy("department").agg(
    F.round(F.avg("work_hours"), 2).alias("avg_work_hours"),
    F.round(F.avg("productivity_score"), 2).alias("avg_productivity"),
    F.round(F.avg("completion_pct"), 2).alias("avg_completion_pct"),
    F.sum("tasks_assigned").alias("total_tasks_assigned"),
    F.sum("tasks_completed").cast("int").alias("total_tasks_completed"))

department_kpis = dept_total.join(dept_issues, on="department", how="left")
department_kpis = department_kpis.join(dept_work, on="department", how="left")
department_kpis = department_kpis.withColumn(
    "absent_rate_pct", F.round(F.col("absent_count") / F.col("total_records") * 100, 2))
department_kpis = department_kpis.withColumn(
    "late_rate_pct", F.round(F.col("late_login_count") / F.col("total_records") * 100, 2))
department_kpis = department_kpis.orderBy(F.col("absent_rate_pct").desc())

display(department_kpis)

department,total_records,absent_count,late_login_count,avg_work_hours,avg_productivity,avg_completion_pct,total_tasks_assigned,total_tasks_completed,absent_rate_pct,late_rate_pct
Engineering,100,16,19,7.76,0.63,79.9,496,394,16.0,19.0
HR,60,9,15,7.61,0.65,78.63,319,249,15.0,25.0
Finance,120,8,26,7.79,0.6,77.31,657,511,6.67,21.67
Sales,120,7,30,7.76,0.59,74.73,684,512,5.83,25.0


In [0]:
# employee level KPIs (useful for finding top absentees and top performers)
emp_total = attendance_clean.groupBy("employee_id", "employee_name", "department").agg(
    F.sum(F.when(F.col("status") == "ABSENT", 1).otherwise(0)).alias("absent_count"),
    F.sum(F.when(F.col("status") == "LATE", 1).otherwise(0)).alias("late_login_count"))

emp_work = worked.groupBy("employee_id").agg(
    F.round(F.avg("work_hours"), 2).alias("avg_work_hours"),
    F.round(F.avg("productivity_score"), 2).alias("avg_productivity"))

employee_kpis = emp_total.join(emp_work, on="employee_id", how="left")
employee_kpis = employee_kpis.orderBy(F.col("absent_count").desc())

print("Top 5 absentees:")
display(employee_kpis.limit(5))

Top 5 absentees:


employee_id,employee_name,department,absent_count,late_login_count,avg_work_hours,avg_productivity
18,Employee 18,Engineering,6,3,7.7,0.66
4,Employee 4,HR,6,6,7.69,0.71
2,Employee 2,Sales,5,2,7.72,0.44
10,Employee 10,Engineering,4,3,7.79,0.65
20,Employee 20,HR,3,5,7.52,0.63


## Step 5: Load - save as Delta tables and export CSV

In [0]:
# save as Delta tables (managed tables in the catalog)
attendance_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("attendance_clean")
department_kpis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("department_kpis")
employee_kpis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("employee_kpis")
print("Delta tables saved: attendance_clean, department_kpis, employee_kpis")

Delta tables saved: attendance_clean, department_kpis, employee_kpis


In [0]:
# export the KPI tables as single csv files
os.makedirs(OUTPUT_PATH, exist_ok=True)
department_kpis.toPandas().to_csv(f"{OUTPUT_PATH}/department_kpis.csv", index=False)
employee_kpis.toPandas().to_csv(f"{OUTPUT_PATH}/employee_kpis.csv", index=False)
print("CSV files saved in:", OUTPUT_PATH)

CSV files saved in: /Volumes/databricks_rr/default/capstone/project1/output


## Step 6: Check the saved table

In [0]:
display(spark.sql("SELECT * FROM department_kpis ORDER BY absent_rate_pct DESC"))

department,total_records,absent_count,late_login_count,avg_work_hours,avg_productivity,avg_completion_pct,total_tasks_assigned,total_tasks_completed,absent_rate_pct,late_rate_pct
Engineering,100,16,19,7.76,0.63,79.9,496,394,16.0,19.0
HR,60,9,15,7.61,0.65,78.63,319,249,15.0,25.0
Finance,120,8,26,7.79,0.6,77.31,657,511,6.67,21.67
Sales,120,7,30,7.76,0.59,74.73,684,512,5.83,25.0
